# Grad-CAM Demo on CIFAR-10
This notebook demonstrates how to compute and visualize Grad-CAM for a CNN model.


## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## 2. Load Model (ResNet18) and CIFAR-10

In [ ]:
# CIFAR-10 normalization
test_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

test_set = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_transforms)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, shuffle=True)

# Pretrained ImageNet ResNet18
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()

# Adapt to CIFAR-10
model.fc = nn.Linear(model.fc.in_features, 10)

## 3. Load a Trained Model (Optional)

In [ ]:
# Uncomment to load your trained model
# model.load_state_dict(torch.load('cifar_resnet18.pth', map_location='cpu'))

## 4. Grad-CAM Hook Helpers

In [ ]:
feature_maps = None
gradients = None

def forward_hook(module, input, output):
    global feature_maps
    feature_maps = output

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

target_layer = model.layer4[1].conv2
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

## 5. Compute Grad-CAM

In [ ]:
def compute_gradcam(input_tensor, class_idx=None):
    global feature_maps, gradients

    output = model(input_tensor)

    if class_idx is None:
        class_idx = output.argmax().item()

    model.zero_grad()
    target = output[0, class_idx]
    target.backward()

    weights = gradients.mean(dim=[1, 2])

    cam = torch.zeros(feature_maps.shape[1:], dtype=torch.float32)
    for i, w in enumerate(weights):
        cam += w * feature_maps[0, i, :, :]

    cam = torch.relu(cam)
    cam -= cam.min()
    cam /= cam.max()

    return cam.detach().numpy(), class_idx

## 6. Visualization

In [ ]:
def show_gradcam(image_tensor, cam, class_name):
    img = image_tensor.squeeze().detach().numpy().transpose(1,2,0)
    img = img * np.array([0.2470,0.2435,0.2616]) + np.array([0.4914,0.4822,0.4465])

    cam_resized = np.interp(np.linspace(0, cam.shape[0], 32), np.arange(cam.shape[0]), cam)
    cam_resized = np.stack([cam_resized]*32).T

    plt.figure(figsize=(6,3))

    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(img)
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title(f"Grad-CAM ({class_name})")
    plt.imshow(img, alpha=0.6)
    plt.imshow(cam_resized, cmap='jet', alpha=0.4)
    plt.axis("off")

    plt.show()

## 7. Run Grad-CAM

In [ ]:
classes = test_set.classes

image, label = next(iter(test_loader))
print("True label:", classes[label[0]])

cam, class_idx = compute_gradcam(image)
print("Predicted label:", classes[class_idx])

show_gradcam(image, cam, classes[class_idx])